# LMA Bonus — No-Positional-Embeddings Ablation (Model H / Hindi)

Optional bonus (assignment PDF "Bonus" section): retrain Model H (Hindi) from
scratch with RoPE disabled entirely (`--no-rope`, `scripts/training/model.py`'s
`GPTConfig.use_rope=False`) — this project's only positional scheme, so this is a
genuine no-position ablation, not a swap to a different scheme. Everything else
(architecture, data, hyperparameters, token budget) is identical to the standard
Hindi model from `kaggle_pretrain.ipynb`, so any difference in the full Phase 2 eval
suite (run separately, locally, once this checkpoint exists) is attributable to
position alone.

**One language only** (Hindi) — this notebook trains a single model on `cuda:0`; the
second GPU in Kaggle's mandatory "GPU T4 x2" shape is simply idle (no single-GPU
option exists on Kaggle).

**Input datasets** (already attached when pushed via the API):

| Dataset | Contents |
| --- | --- |
| `sohailmemon/lma-individual` | `hindi/{train,val,test}.txt`, `hindi/stats.json` (optionally `tokenized/hindi/...`) |
| `sohailmemon/lma-individual-files` | `hindi/tokenizer/` |
| `sohailmemon/lma-bonus-scripts` | fresh `scripts/` tree including the `--no-rope` flag |
| `sohailmemon/lma-secrets` (private) | `secrets.json` with `HF_TOKEN`/`WANDB_API_KEY` - read directly since secrets attached via the Kaggle website's Secrets UI don't carry over when a kernel is pushed via the API |

Pushes to a **separate** HF repo (`sohailtsm/lma-hindi-gpt-noposition`) and a
separate W&B project, so this never collides with the real Hindi model
(`sohailtsm/lma-hindi-gpt`). Same 12h-session-cap / auto-resume-from-HF handling as
`kaggle_pretrain.ipynb` — full scale (~28k iters) is expected to need 2 sessions.


## 0. Setup

In [ ]:
!pip install -q sentencepiece wandb huggingface_hub


In [ ]:
import shutil
from pathlib import Path


def find_input(slug: str, owner: str = "sohailmemon") -> Path:
    """Kaggle's dataset mount path convention has varied across notebooks in this
    project (plain slug vs. owner-prefixed); try both rather than guessing."""
    candidates = [
        Path(f"/kaggle/input/{slug}"),
        Path(f"/kaggle/input/datasets/{owner}/{slug}"),
    ]
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError(f"none of {candidates} exist - is '{slug}' added as an input dataset?")


FILES_DS = find_input("lma-individual-files")
DATA_DS = find_input("lma-individual")
SCRIPTS_DS = find_input("lma-bonus-scripts")
WORK = Path("/kaggle/working")

if (WORK / "scripts").exists():
    shutil.rmtree(WORK / "scripts")
shutil.copytree(SCRIPTS_DS, WORK / "scripts")

import sys
sys.path.insert(0, str(WORK))
print("FILES_DS:", FILES_DS)
print("DATA_DS:", DATA_DS)
print("SCRIPTS_DS:", SCRIPTS_DS, "-> copied to", WORK / "scripts")


In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
n_gpu = torch.cuda.device_count()
print("gpu count:", n_gpu)
for i in range(n_gpu):
    print(f"  cuda:{i}", torch.cuda.get_device_name(i))


### Logging secrets

In [ ]:
import json
import os

SECRETS_DS = find_input("lma-secrets")
secrets = json.loads((SECRETS_DS / "secrets.json").read_text())
os.environ["HF_TOKEN"] = secrets["HF_TOKEN"]
os.environ["WANDB_API_KEY"] = secrets["WANDB_API_KEY"]
print("loaded HF_TOKEN and WANDB_API_KEY from", SECRETS_DS / "secrets.json")

assert os.environ.get("WANDB_API_KEY"), "secrets.json is missing WANDB_API_KEY"
assert os.environ.get("HF_TOKEN"), "secrets.json is missing HF_TOKEN"


## 1. Config

Identical to `kaggle_pretrain.ipynb`'s Hindi config except: `BATCH_SIZE=32`,
`GRAD_ACCUM_STEPS=2` (effective batch 64 either way - `train.py` already normalizes
for this by dividing the loss by `grad_accum_steps` before `.backward()`, so this is
a throughput/memory choice only, not a change to training dynamics), and `NO_ROPE =
True`.


In [ ]:
LANG = "hindi"

BATCH_SIZE = 32          # micro-batch
GRAD_ACCUM_STEPS = 2     # effective batch size = BATCH_SIZE * GRAD_ACCUM_STEPS = 64
BLOCK_SIZE = 512
EPOCHS = 2               # same as the standard Hindi run
NO_ROPE = True

WARMUP_ITERS = 300
LR = 3e-4
MIN_LR = 3e-5
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0

EVAL_INTERVAL = 500
EVAL_ITERS = 100
LOG_INTERVAL = 50
CHECKPOINT_INTERVAL = 1000

WANDB_PROJECT = "lma-individual-pretrain"

HF_USERNAME = "sohailtsm"
HF_REPO = f"{HF_USERNAME}/lma-{LANG}-gpt-noposition"
HF_PUBLIC = True

# Auto-filled below by checking HF for an existing last.pt. Leave as None here; set to
# False to force a from-scratch run instead.
RESUME = None


### Auto-resume from the Hugging Face Hub

Same pattern as `kaggle_pretrain.ipynb` - re-running this whole notebook after a
crash or Kaggle session timeout picks up training where it left off automatically.


In [ ]:
from huggingface_hub import hf_hub_download

if RESUME is not False:
    try:
        RESUME = hf_hub_download(HF_REPO, "last.pt")
        print(f"found existing checkpoint on {HF_REPO}, will resume from it")
    except Exception as e:
        print(f"no existing checkpoint on {HF_REPO} ({type(e).__name__}); starting from scratch")
        RESUME = None


## 2. Tokenize (or reuse pre-tokenized data)

Same as `kaggle_pretrain.ipynb` §2 - checks for pre-tokenized `lma-individual`
contents first, else runs `prepare_bin.py`.


In [ ]:
import subprocess

pretokenized = DATA_DS / "tokenized" / LANG
if (pretokenized / "train.bin").exists() and (pretokenized / "tokenized_meta.json").exists():
    print(f"found pre-tokenized data in {pretokenized}, skipping prepare_bin.py")
    BIN_DIR = pretokenized
else:
    BIN_DIR = WORK / LANG / "data"
    BIN_DIR.mkdir(parents=True, exist_ok=True)
    cmd = [
        "python", "-u", "-m", "scripts.training.prepare_bin",
        "--lang", LANG,
        "--data-dir", str(DATA_DS / LANG),
        "--tokenizer", str(FILES_DS / LANG / "tokenizer" / f"{LANG}_unigram_16k.model"),
        "--out-dir", str(BIN_DIR),
    ]
    print(f"=== tokenizing {LANG} ===")
    subprocess.run(cmd, cwd=WORK, check=True)


In [ ]:
import json

meta = json.loads((BIN_DIR / "tokenized_meta.json").read_text())
train_tokens = meta["tokens"]["train"]
tokens_per_iter = BATCH_SIZE * GRAD_ACCUM_STEPS * BLOCK_SIZE
max_iters = max(1, (train_tokens * EPOCHS) // tokens_per_iter)
print(f"{LANG}: {train_tokens:,} train tokens -> {max_iters:,} iters for {EPOCHS} epoch(s) "
      f"({tokens_per_iter:,} tokens/iter)")


## 3. Pretrain (no positional embeddings)

Single model on `cuda:0`. `--no-rope` is the only architectural difference from the
standard Hindi pretraining run.


In [ ]:
import os
import time

out_dir = WORK / LANG / "ablation_checkpoints"
out_dir.mkdir(parents=True, exist_ok=True)
log_path = WORK / f"{LANG}_ablation_train.log"

cmd = [
    "python", "-u", "-m", "scripts.training.train",
    "--lang", LANG,
    "--data-dir", str(BIN_DIR),
    "--out-dir", str(out_dir),
    "--device", "cuda:0",
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum-steps", str(GRAD_ACCUM_STEPS),
    "--block-size", str(BLOCK_SIZE),
    "--max-iters", str(max_iters),
    "--warmup-iters", str(WARMUP_ITERS),
    "--lr", str(LR),
    "--min-lr", str(MIN_LR),
    "--weight-decay", str(WEIGHT_DECAY),
    "--grad-clip", str(GRAD_CLIP),
    "--eval-interval", str(EVAL_INTERVAL),
    "--eval-iters", str(EVAL_ITERS),
    "--log-interval", str(LOG_INTERVAL),
    "--checkpoint-interval", str(CHECKPOINT_INTERVAL),
    "--wandb-project", WANDB_PROJECT,
    "--wandb-run-name", f"{LANG}-gpt-noposition",
    "--hf-repo-id", HF_REPO,
]
if NO_ROPE:
    cmd.append("--no-rope")
if HF_PUBLIC:
    cmd.append("--hf-public")
if RESUME:
    cmd += ["--resume", RESUME]

env = os.environ | {"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
log_f = open(log_path, "w")
proc = subprocess.Popen(cmd, cwd=WORK, stdout=log_f, stderr=subprocess.STDOUT, env=env)

position = 0
while proc.poll() is None:
    with open(log_path) as f:
        f.seek(position)
        new_text = f.read()
        position = f.tell()
    for line in new_text.splitlines():
        print(line)
    time.sleep(5.0)
with open(log_path) as f:
    f.seek(position)
    for line in f.read().splitlines():
        print(line)
print(f"process exited with code {proc.returncode}")
if proc.returncode != 0:
    raise RuntimeError(f"training failed with exit code {proc.returncode} - see {log_path}")


## 4. Summary

- **Checkpoints**: `/kaggle/working/hindi/ablation_checkpoints/{best,last}.pt`.
- **Weights & Biases**: `https://wandb.ai/<your-entity>/lma-individual-pretrain`, run
  `hindi-gpt-noposition`.
- **Hugging Face Hub**: `https://huggingface.co/sohailtsm/lma-hindi-gpt-noposition`
  (public) - `best.pt`, `last.pt`, `config.json` (with `use_rope: false`).

Full Phase 2 evaluation suite (PPL/BPB, generation quality, attention analysis) is
run **separately, locally** against this checkpoint once training finishes - see
`report/bonus_ablation_report.md`.


In [ ]:
ckpt_dir = WORK / LANG / "ablation_checkpoints"
if ckpt_dir.exists():
    for f in sorted(ckpt_dir.iterdir()):
        print(f"  {f.name}: {f.stat().st_size / 1e6:.1f} MB")
print(f"  HF repo: https://huggingface.co/{HF_REPO}")
